# Phase 2 & 3: BBSE Label Shift + SAR Test-Time Adaptation
### Fashion-MNIST | Loads Phase 1 DivideMix Weights

**Pipeline Overview:**
- **Phase 2:** BBSE (Black-Box Shift Estimation) to recover target class frequencies ŵ_t via confusion matrix + least-squares
- **Phase 3A:** BatchNorm statistics update on target stream (zero backprop)
- **Phase 3B:** SAR — Sharpness-Aware Reliable Entropy Minimisation using ŵ_t

**Required files:**
- `best_model_phase1.pt` — Phase 1 model state dict
- `val_sanity.pt` — small clean validation set
- `static.pt` — shifted + imbalanced target stream

## 0. Imports & Setup

In [ ]:
import os
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = 10

CLASS_NAMES = [
    'T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot',
]

print(f'Using device: {DEVICE}')

## 1. Paths & Config

In [ ]:
import shutil
from google.colab import drive
drive.mount('/content/drive')

# ── Drive paths ───────────────────────────────────────────────────────────────
DRIVE_SAVE_DIR = '/content/drive/MyDrive/hackenza_results'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'All artifacts will be saved to: {DRIVE_SAVE_DIR}')

PHASE1_WEIGHTS = os.path.join(DRIVE_SAVE_DIR, 'best_model_phase1.pt')
VAL_PATH       = '/content/drive/MyDrive/val_sanity.pt'
TARGET_PATH    = '/content/drive/MyDrive/static.pt'

print(f'Phase 1 weights : {PHASE1_WEIGHTS}')
print(f'Val data        : {VAL_PATH}')
print(f'Target data     : {TARGET_PATH}')

## 2. Dataset Class

In [ ]:
class TorchDataset(Dataset):
    """Generic wrapper for .pt files with images/labels."""

    def __init__(self, filepath, transform=None):
        payload = torch.load(filepath, weights_only=False)

        if 'images' in payload:
            self.data = payload['images']
        elif 'data' in payload:
            self.data = payload['data']
        else:
            raise KeyError(f"No 'images'/'data' key. Keys: {list(payload.keys())}")

        if 'labels' in payload:
            self.labels = payload['labels'].long()
        elif 'targets' in payload:
            self.labels = payload['targets'].long()
        elif 'label' in payload:
            self.labels = payload['label'].long()
        else:
            print(f'  Warning: no label key in {filepath}. Using dummy -1.')
            self.labels = torch.full((self.data.shape[0],), -1, dtype=torch.long)

        self.transform = transform

        if self.data.dim() == 3:
            self.data = self.data.unsqueeze(1)
        if self.data.dtype != torch.float32:
            self.data = self.data.float()
        if self.data.max() > 1.0:
            self.data = self.data / 255.0

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.data[idx]
        lbl = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, lbl


print('TorchDataset defined.')

## 3. Model: ResNet-18 (Modified for 28×28 Grayscale)

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class ResNet18_FMNIST(nn.Module):
    """
    ResNet-18 adapted for Fashion-MNIST (28x28 grayscale).
    Key changes vs standard ResNet-18:
      - Input conv: 3x3 kernel, stride 1 (not 7x7 stride 2)
      - No initial MaxPool
      - Modified head: Linear -> BN -> ReLU -> Dropout(0.3) -> Linear
    """
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
        self._initialize_weights()

    def _make_layer(self, in_planes, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(in_planes, planes, s))
            in_planes = planes
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_uniform_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.head(x)


# Quick sanity check
dummy = torch.zeros(2, 1, 28, 28)
model_test = ResNet18_FMNIST()
out = model_test(dummy)
print(f'Output shape: {out.shape}  (expected: [2, 10])')
total_params = sum(p.numel() for p in model_test.parameters())
print(f'Total parameters: {total_params:,}')

## 4. Load Phase 1 Model & Data

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────
FMNIST_MEAN = (0.2860,)
FMNIST_STD  = (0.3530,)

eval_transform = transforms.Compose([
    transforms.Normalize(FMNIST_MEAN, FMNIST_STD),
])

# ── Load Phase 1 model ────────────────────────────────────────────────────────
model = ResNet18_FMNIST(NUM_CLASSES).to(DEVICE)
state_dict = torch.load(PHASE1_WEIGHTS, map_location=DEVICE, weights_only=False)
model.load_state_dict(state_dict)
model.eval()
print(f'Phase 1 model loaded from {PHASE1_WEIGHTS}')

# ── Load datasets ─────────────────────────────────────────────────────────────
val_dataset    = TorchDataset(VAL_PATH,    transform=eval_transform)
target_dataset = TorchDataset(TARGET_PATH, transform=eval_transform)

val_loader    = DataLoader(val_dataset,    batch_size=64, shuffle=False, num_workers=0)
target_loader = DataLoader(target_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f'Val (sanity)    samples: {len(val_dataset)}')
print(f'Target (static) samples: {len(target_dataset)}')

## 5. Utility: Evaluate Accuracy

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    """Evaluate model accuracy on a labeled loader."""
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return correct / total if total > 0 else 0.0


# Baseline accuracy (Phase 1 model, no adaptation)
acc_baseline = evaluate(model, target_loader, DEVICE)
print(f'Baseline accuracy on target (no TTA): {acc_baseline:.4f}')

---
## 6. Phase 2 — BBSE Label Shift Estimation

**Algorithm:**
1. Compute confusion matrix $C$ on `val_sanity.pt`: $C[i][j] = P(\hat{y}=i \mid y=j)$
2. Get average softmax predictions on the unlabeled target stream: $\mu_{\mathrm{target}} = \frac{1}{N}\sum_i \mathrm{softmax}(f(x_i))$
3. Solve the linear system: $C^\top \hat{w}_t = \mu_{\mathrm{target}}$ via least-squares
4. Post-process: $\hat{w}_t = \mathrm{clip}(\hat{w}_t, \min=0)$, then normalise

### 6.1 Step 1 — Confusion Matrix on val_sanity.pt

In [ ]:
@torch.no_grad()
def compute_confusion_matrix(model, loader, num_classes, device):
    """
    Build confusion matrix C where:
        C[i][j] = P(predict class i | true class j)
    Uses hard predictions (argmax) to estimate the conditional.
    """
    model.eval()
    # counts[i][j] = number of times true=j was predicted as i
    counts = torch.zeros(num_classes, num_classes, dtype=torch.float64)

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        for pred, true in zip(preds.cpu(), labels.cpu()):
            counts[pred.item(), true.item()] += 1

    # Normalize columns: C[:, j] = counts[:, j] / counts[:, j].sum()
    col_sums = counts.sum(dim=0, keepdim=True).clamp(min=1)
    C = counts / col_sums
    return C


C = compute_confusion_matrix(model, val_loader, NUM_CLASSES, DEVICE)
print(f'Confusion matrix C shape: {C.shape}')
print(f'Column sums (should be ~1): {C.sum(dim=0).numpy().round(4)}')

In [ ]:
# ── Visualize confusion matrix ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(C.numpy(), cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(CLASS_NAMES, fontsize=8)
ax.set_xlabel('True class j')
ax.set_ylabel('Predicted class i')
ax.set_title('Confusion Matrix C: P(pred=i | true=j)')
plt.colorbar(im, ax=ax, shrink=0.8)

# Annotate cells
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        val = C[i, j].item()
        color = 'white' if val > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=7, color=color)

plt.tight_layout()
plt.show()

### 6.2 Step 2 — Average Softmax on Target Stream

In [ ]:
@torch.no_grad()
def compute_avg_softmax(model, loader, num_classes, device):
    """mu_target = (1/N) * sum softmax(model(x_i))   shape: [K]"""
    model.eval()
    total_probs = torch.zeros(num_classes, dtype=torch.float64)
    n_samples = 0
    for images, _ in loader:
        images = images.to(device)
        probs = F.softmax(model(images), dim=1)  # (B, K)
        total_probs += probs.sum(dim=0).cpu().double()
        n_samples += images.size(0)
    return total_probs / n_samples


mu_target = compute_avg_softmax(model, target_loader, NUM_CLASSES, DEVICE)
print(f'mu_target (avg softmax on target):')
for name, mu in zip(CLASS_NAMES, mu_target.numpy()):
    print(f'  {name:12s}: {mu:.6f}')

### 6.3 Step 3 — Solve Linear System: $C^\top \hat{w}_t = \mu_{\mathrm{target}}$

In [ ]:
C_np  = C.numpy()
mu_np = mu_target.numpy()

# Solve: C^T @ w_hat = mu_target  via least-squares
w_hat, residuals, rank, sv = np.linalg.lstsq(C_np.T, mu_np, rcond=None)

print(f'Raw BBSE solution w_hat: {w_hat.round(6)}')
print(f'  lstsq rank: {rank}')
print(f'  residuals : {residuals}')

### 6.4 Step 4 — Post-process $\hat{w}_t$

In [ ]:
# Clip negatives, then normalise to a valid probability distribution
w_hat = np.clip(w_hat, a_min=0, a_max=None)
w_hat = w_hat / w_hat.sum()

print('Post-processed w_hat (clipped & normalized):')
for name, w in zip(CLASS_NAMES, w_hat):
    print(f'  {name:12s}: w_hat = {w:.6f}')

w_hat_tensor = torch.tensor(w_hat, dtype=torch.float32)
torch.save(w_hat_tensor, 'bbse_label_shift_weights.pt')
print(f'\nSaved BBSE weights to bbse_label_shift_weights.pt')

In [ ]:
# ── Visualize BBSE estimated distribution ─────────────────────────────────────
source_freq = np.ones(NUM_CLASSES) / NUM_CLASSES  # uniform source

x = np.arange(NUM_CLASSES)
width = 0.35
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - width/2, source_freq, width, label='Source p_s(y) [uniform]', alpha=0.8, color='steelblue')
ax.bar(x + width/2, w_hat,       width, label='Target w_hat [BBSE est.]', alpha=0.8, color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_title('Label Shift: Source vs BBSE-Estimated Target Distribution')
ax.set_ylabel('Probability')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('phase2_bbse_label_shift.png', dpi=120, bbox_inches='tight')
plt.show()
print('Label shift plot saved.')

---
## 7. Phase 3 — Two-Step Test-Time Adaptation

**Step A:** Update BatchNorm running statistics on the target stream (forward pass only, no backprop).

**Step B:** SAR — freeze all params except BN affine weights, adjust logits by $\log(\hat{w}_t)$, minimise entropy on uncertain samples.

### 7.1 Step A — BatchNorm Statistics Update

In [ ]:
# Work on a deep copy so the original Phase 1 model stays intact for comparison
adapted_model = copy.deepcopy(model).to(DEVICE)

# Reset BN running stats so they are recomputed from scratch on target data
for m in adapted_model.modules():
    if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
        m.reset_running_stats()

# BN layers update running_mean / running_var in train mode
adapted_model.train()
with torch.no_grad():
    for images, _ in target_loader:
        images = images.to(DEVICE)
        _ = adapted_model(images)  # forward pass only — BN stats update automatically

adapted_model.eval()
print('BN running stats updated on target distribution (zero backpropagation).')

# ── Evaluate after BN update ──────────────────────────────────────────────────
acc_bn_only = evaluate(adapted_model, target_loader, DEVICE)
print(f'\nBaseline accuracy (no TTA)       : {acc_baseline:.4f}')
print(f'After BN stats update (Step A)   : {acc_bn_only:.4f}')
print(f'BN update improvement            : +{(acc_bn_only - acc_baseline)*100:.2f}%')

### 7.2 Step B — SAR: Sharpness-Aware Reliable Entropy Minimisation

In [ ]:
# ── SAR hyperparameters ───────────────────────────────────────────────────────
SAR_LR            = 5e-4
ENTROPY_THRESHOLD = 0.4 * np.log(NUM_CLASSES)  # ~0.921
SAR_EPOCHS        = 1  # single pass over target stream

print(f'SAR learning rate      : {SAR_LR}')
print(f'Entropy threshold      : {ENTROPY_THRESHOLD:.4f}  (0.4 * ln({NUM_CLASSES}))')
print(f'SAR epochs             : {SAR_EPOCHS}')

In [ ]:
# ── Freeze everything except BN affine parameters ─────────────────────────────
bn_params = []
for module in adapted_model.modules():
    if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
        module.requires_grad_(True)
        bn_params.extend([module.weight, module.bias])

for name, param in adapted_model.named_parameters():
    if not any(param is p for p in bn_params):
        param.requires_grad_(False)

optimizer = optim.SGD(bn_params, lr=SAR_LR, momentum=0.9)
print(f'Adapting {len(bn_params)} BN parameters')

# Log(w_hat) for Bayes-optimal logit adjustment under label shift
log_w_hat = torch.log(w_hat_tensor.to(DEVICE) + 1e-8)
print(f'log(w_hat): {log_w_hat.cpu().numpy().round(4)}')

In [ ]:
# ── SAR adaptation loop ───────────────────────────────────────────────────────
entropy_log = []

for sar_epoch in range(1, SAR_EPOCHS + 1):
    adapted_model.train()
    # Keep BatchNorm1d in eval mode to avoid issues with small batches
    for module in adapted_model.modules():
        if isinstance(module, nn.BatchNorm1d):
            module.eval()

    total_loss = 0.0
    n_adapted_batches = 0

    for images, _ in target_loader:
        images = images.to(DEVICE)

        logits = adapted_model(images)

        # Adjust logits by log of class priors (Phase 2 output)
        # Bayes-optimal calibration under label shift
        logits_adjusted = logits + log_w_hat

        probs = F.softmax(logits_adjusted, dim=1)
        entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1)  # per-sample entropy

        # Only adapt on uncertain predictions — confident ones are likely already correct
        uncertain_mask = entropy > ENTROPY_THRESHOLD

        if uncertain_mask.sum() <= 1:
            # Skip if no (or too few) uncertain samples
            entropy_log.append(entropy.mean().item())
            continue

        loss = entropy[uncertain_mask].mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_adapted_batches += 1
        entropy_log.append(entropy.mean().item())

    avg_loss = total_loss / max(n_adapted_batches, 1)
    print(f'SAR epoch {sar_epoch}: avg entropy loss = {avg_loss:.4f}, '
          f'adapted batches = {n_adapted_batches}/{len(target_loader)}')

adapted_model.eval()
print('SAR adaptation complete.')

In [ ]:
# ── Entropy over adaptation stream ────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(entropy_log, alpha=0.7, color='purple')
plt.axhline(ENTROPY_THRESHOLD, color='red', linestyle='--',
            label=f'Entropy gate ({ENTROPY_THRESHOLD:.3f})')
plt.xlabel('Batch index')
plt.ylabel('Mean batch entropy')
plt.title('SAR: Entropy over Target Stream (lower = more confident)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('phase3_entropy.png', dpi=120, bbox_inches='tight')
plt.show()
print('Entropy plot saved.')

---
## 8. Results Summary

In [ ]:
acc_sar = evaluate(adapted_model, target_loader, DEVICE)

print('=' * 60)
print('           PIPELINE RESULTS SUMMARY')
print('=' * 60)
print(f'  Phase 2 method         : BBSE (confusion matrix + lstsq)')
print(f'  Phase 3A method        : BatchNorm stats update')
print(f'  Phase 3B method        : SAR (entropy minimisation with w_hat)')
print('-' * 60)
print(f'  Baseline (no TTA)      : {acc_baseline:.4f}')
print(f'  After BN update only   : {acc_bn_only:.4f}  (+{(acc_bn_only - acc_baseline)*100:.2f}%)')
print(f'  After SAR (Step A + B) : {acc_sar:.4f}  (+{(acc_sar - acc_baseline)*100:.2f}%)')
print('-' * 60)
print(f'  BBSE estimated target distribution w_hat:')
for name, w in zip(CLASS_NAMES, w_hat):
    print(f'    {name:12s}: {w:.4f}')
print('=' * 60)

## 9. Save Final Artifacts

In [ ]:
# ── Save adapted model ────────────────────────────────────────────────────────
torch.save(adapted_model.state_dict(), 'sar_adapted_model.pt')
print('Adapted model saved to sar_adapted_model.pt')

# ── Save full pipeline artifact ───────────────────────────────────────────────
torch.save({
    'model_state_dict'  : adapted_model.state_dict(),
    'bbse_weights'      : w_hat_tensor,
    'confusion_matrix'  : C,
    'mu_target'         : mu_target,
    'config': {
        'num_classes'       : NUM_CLASSES,
        'sar_lr'            : SAR_LR,
        'entropy_threshold' : ENTROPY_THRESHOLD,
        'sar_epochs'        : SAR_EPOCHS,
    },
    'results': {
        'baseline_acc'  : acc_baseline,
        'bn_update_acc' : acc_bn_only,
        'sar_acc'       : acc_sar,
    },
}, 'final_pipeline_phase2_3.pt')
print('Full pipeline artifact saved to final_pipeline_phase2_3.pt')

---
## 10. Generate submission.csv

Predict on `target_static.pt` (static_i rows) and `test_suite_public.pt` (scenario_XX_i rows), then build the final CSV.

In [ ]:
import csv

# ══════════════════════════════════════════════════════════════════════════════
# Helper: Run BN-update + SAR on a fresh copy of the Phase 1 model for a batch
# of images, then return predictions. This mirrors the per-scenario adaptation
# from the original pipeline.
# ══════════════════════════════════════════════════════════════════════════════

def adapt_and_predict_scenario(base_model, images_normed, w_hat_t, device,
                                num_classes=10, sar_lr=5e-4,
                                entropy_threshold=None):
    """
    Given normalised images (N,1,28,28), run Phase 3 (BN update + SAR)
    on a fresh deep-copy and return hard predictions (N,).
    """
    if entropy_threshold is None:
        entropy_threshold = 0.4 * np.log(num_classes)

    log_w = torch.log(w_hat_t.to(device) + 1e-8)
    mdl = copy.deepcopy(base_model).to(device)

    # ── Step A: BN stats update ───────────────────────────────────────────────
    for m in mdl.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            m.reset_running_stats()
    mdl.train()
    loader = DataLoader(
        torch.utils.data.TensorDataset(images_normed),
        batch_size=64, shuffle=False
    )
    with torch.no_grad():
        for (batch,) in loader:
            _ = mdl(batch.to(device))
    mdl.eval()

    # ── Step B: SAR entropy minimisation ──────────────────────────────────────
    bn_p = []
    for module in mdl.modules():
        if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
            module.requires_grad_(True)
            bn_p.extend([module.weight, module.bias])
    for p in mdl.parameters():
        if not any(p is bp for bp in bn_p):
            p.requires_grad_(False)

    opt = optim.SGD(bn_p, lr=sar_lr, momentum=0.9)

    mdl.train()
    for module in mdl.modules():
        if isinstance(module, nn.BatchNorm1d):
            module.eval()

    for (batch,) in loader:
        batch = batch.to(device)
        logits = mdl(batch) + log_w
        probs = F.softmax(logits, dim=1)
        ent = -(probs * torch.log(probs + 1e-8)).sum(dim=1)
        mask = ent > entropy_threshold
        if mask.sum() <= 1:
            continue
        loss = ent[mask].mean()
        opt.zero_grad()
        loss.backward()
        opt.step()

    # ── Predict ───────────────────────────────────────────────────────────────
    mdl.eval()
    all_preds = []
    with torch.no_grad():
        for (batch,) in loader:
            preds = mdl(batch.to(device)).argmax(dim=1).cpu()
            all_preds.append(preds)
    return torch.cat(all_preds)


print('adapt_and_predict_scenario() defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PART A: Predict on target_static.pt  (IDs: static_0, static_1, …)
# ══════════════════════════════════════════════════════════════════════════════
print('Predicting on target_static (for static_i rows)...')

# Use the already-adapted model for static predictions
adapted_model.eval()
static_preds = []
with torch.no_grad():
    for images, _ in target_loader:
        images = images.to(DEVICE)
        preds = adapted_model(images).argmax(dim=1).cpu()
        static_preds.append(preds)
static_preds = torch.cat(static_preds)
print(f'  Static predictions: {len(static_preds)} samples')

# Save to Drive
torch.save(static_preds, 'static_preds.pt')
shutil.copy('static_preds.pt', os.path.join(DRIVE_SAVE_DIR, 'static_preds.pt'))
print('  Static predictions saved to Drive')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PART B: Predict on test_suite_public.pt  (IDs: scenario_XX_0, scenario_XX_1, …)
# ══════════════════════════════════════════════════════════════════════════════
TEST_SUITE_PATH = '/content/drive/MyDrive/test_suite_public.pt'
test_suite = torch.load(TEST_SUITE_PATH, weights_only=False)
print(f'Test suite loaded: {len(test_suite)} scenarios')
print(f'Scenario keys: {sorted(test_suite.keys())}\n')

all_predictions = {}

for scenario_name in sorted(test_suite.keys()):
    scenario_data = test_suite[scenario_name]

    if isinstance(scenario_data, dict):
        images = scenario_data['images']
    else:
        images = scenario_data

    if images.dim() == 3:
        images = images.unsqueeze(1)
    if images.dtype != torch.float32:
        images = images.float()
    if images.max() > 1.0:
        images = images / 255.0

    # Normalise with same stats as training
    images_normed = (images - FMNIST_MEAN[0]) / FMNIST_STD[0]

    # Per-scenario adaptation (fresh BN update + SAR)
    scenario_preds = adapt_and_predict_scenario(
        base_model=model,            # original Phase 1 model
        images_normed=images_normed,
        w_hat_t=w_hat_tensor,
        device=DEVICE,
        num_classes=NUM_CLASSES,
    )
    all_predictions[scenario_name] = scenario_preds

    class_counts = torch.bincount(scenario_preds, minlength=NUM_CLASSES)
    print(f'  {scenario_name}: {len(images)} imgs, '
          f'top class: {CLASS_NAMES[class_counts.argmax().item()]}')

# Save to Drive
torch.save(all_predictions, 'all_scenario_predictions.pt')
shutil.copy('all_scenario_predictions.pt',
            os.path.join(DRIVE_SAVE_DIR, 'all_scenario_predictions.pt'))
print('\nAll scenario predictions saved to Drive')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PART C: Build submission.csv  —  columns: ID, Category
# ══════════════════════════════════════════════════════════════════════════════
csv_path = 'submission.csv'
row_count = 0

with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['ID', 'Category'])

    # static_i rows
    for i, pred in enumerate(static_preds):
        writer.writerow([f'static_{i}', pred.item()])
        row_count += 1

    # scenario_XX_i rows
    for scenario_name in sorted(all_predictions.keys()):
        preds = all_predictions[scenario_name]
        for i, pred in enumerate(preds):
            writer.writerow([f'{scenario_name}_{i}', pred.item()])
            row_count += 1

print(f'\nsubmission.csv saved — {row_count} rows')
print(f'   static rows   : {len(static_preds)}')
print(f'   scenario rows : {sum(len(p) for p in all_predictions.values())}')

# Save to Drive
shutil.copy(csv_path, os.path.join(DRIVE_SAVE_DIR, 'submission.csv'))
print(f'   submission.csv copied to Drive')

# ── Download ──────────────────────────────────────────────────────────────────
from google.colab import files
files.download(csv_path)